In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT")

In [2]:
os.environ["CHROMA_API_KEY"] = os.getenv("CHROMA_API_KEY")
os.environ["CHROMA_TENANT"] = os.getenv("CHROMA_TENANT")
os.environ['CHROMA_DATABASE'] = os.getenv("CHROMA_DATABASE")

In [3]:
## Data Ingestion to scrape the website
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from chromadb.config import Settings

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [5]:
embeddings = OllamaEmbeddings(model=os.getenv("OLLAMA_MODEL"))

In [7]:
import chromadb
# Connect to Chroma Cloud
client = chromadb.CloudClient(
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
    api_key=os.getenv("CHROMA_API_KEY")
)
# collection = client.create_collection(name="simpleapp-collection")

In [8]:
client.get_collection("simpleapp-collection").count()

0

In [9]:
settings = Settings(
    chroma_server_host="api.trychroma.com",
    chroma_server_http_port=8000,
    anonymized_telemetry=False,
    tenant_id=os.getenv("CHROMA_TENANT")
)

vectorstore = Chroma(
    collection_name="simpleapp-collection",
    embedding_function=embeddings,
    client_settings=settings,
    chroma_cloud_api_key=os.getenv("CHROMA_API_KEY"),
    database=os.getenv("CHROMA_DATABASE"),
    tenant=os.getenv("CHROMA_TENANT")
)

#### Loading the web data

In [15]:
loader=WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
loader

In [16]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTutorialPolly AI assistantTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run 

In [17]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTutorialPolly AI assistantTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run 

In [18]:
len(documents)

18

In [19]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(documents))]

In [20]:
uuids

['320b1fac-64eb-4b49-80af-378d3300f0ca',
 'd383bb54-5d68-4afe-83b4-cfd476819bbb',
 'aead21a1-7b1e-44e3-8f9a-cc1fcd9a7eaa',
 'fc2892a0-9ed4-4647-bdd9-8285456caf33',
 '4894a320-af04-4fbf-bd21-54d41dd76b0b',
 '5e0d4063-8cbe-40d0-9cfb-e70cf7f7d94d',
 '3856860d-89aa-4752-b465-f33e7e15bf49',
 '441ac885-9816-4b09-8e25-884060abac5f',
 'edfee64a-3a73-4681-a708-6f4cffc99997',
 'e72ec1da-6fcd-4098-8d49-fd194cdad637',
 '62b2f6d7-de05-45e8-9c21-c9855930a8ba',
 '82b11eb2-7cc4-40a7-a2fb-3f4129394cd4',
 'cb53a7df-07ee-46fb-af1e-53576d5c4085',
 '74d5e086-c14d-4f86-ad99-20919756d710',
 '8a8500d0-87b6-4dd3-b6eb-7b5625578484',
 'c5862df2-974f-41f8-8b45-33a5205fcf82',
 'f36e925c-e2b7-4b60-97e8-d0c5ba7246d7',
 '769fb15a-bb37-4311-8390-53bf7497e0b2']

In [21]:
vectorstore.add_documents(documents, id=uuids)

['1290684e-7568-4a23-8323-3a05b00e95ac',
 'f3cb3613-4a1b-457b-a1b6-e700a663f8a7',
 '52d1d137-7f49-4cb3-a577-4061c6bd752f',
 '66886143-5f8c-4440-bdd8-5dc813e5406d',
 'a5523b66-7ea3-4d14-ad79-539fa7a23651',
 '58e6492e-5805-4b78-9739-9e3bf69ba468',
 '32a3b4da-66c0-42f2-a9bc-df3a9ece128f',
 'd6ca4367-df56-4389-a4bc-76899fb00e29',
 '42ae6d09-41ce-47cb-bef2-4261b1c8be1a',
 '8bb1d0ff-a818-4612-aad3-d2d28295035e',
 '40967ea1-fc1a-45b8-ae5b-ab2ce38729e5',
 '232e146d-956e-4ac6-abaf-17669e206542',
 '3f4d7da5-766f-49e3-9d8e-6430d20f8576',
 'e8aa3f6b-5930-442a-90e8-b4a2d14580ff',
 'a003790d-f1f9-4ff0-a712-d4dadba64bc5',
 'e0411c73-4ae5-4a58-b762-dbcc2661746b',
 'a3af3819-d653-4e03-a00e-4c713702895e',
 '7e836467-f81a-4a5d-a6bf-a09b8954446e']

In [22]:
vectorstore

In [23]:
query = "What benefit does having observability set up from the start has and how can it be set up"
result = vectorstore.similarity_search(query)
print(result)

[Document(id='a3af3819-d653-4e03-a00e-4c713702895e', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en', 'title': 'Trace an LLM application tutorial - Docs by LangChain'}, page_content='\u200bA/B testing\nGroup-by functionality requires at least two different values for a given metadata key.\nBecause you have been logging the llm metadata attribute, you can group monitoring charts by that attribute to compare model performance over time. From Monitoring in the UI sidebar, click Group by in the top left corner, select Metadata from the dropdown, then select llm. The charts update to show results grouped by that attribute. For more on grouping and custom charts, refer to Dashboards.\n\u200bDrilldown\nWhen a monitoring chart shows something unexpected, click a data point to freeze the tooltip, then click the metric name 

In [24]:
print(result[0].page_content)

​A/B testing
Group-by functionality requires at least two different values for a given metadata key.
Because you have been logging the llm metadata attribute, you can group monitoring charts by that attribute to compare model performance over time. From Monitoring in the UI sidebar, click Group by in the top left corner, select Metadata from the dropdown, then select llm. The charts update to show results grouped by that attribute. For more on grouping and custom charts, refer to Dashboards.
​Drilldown
When a monitoring chart shows something unexpected, click a data point to freeze the tooltip, then click the metric name (for example, Input) to jump to the filtered runs table for that time window. For more on searching and filtering runs, refer to Filter traces.


In [30]:
from langchain_core.prompts import ChatPromptTemplate


In [39]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below:

<context>
{context}
</context>
                                          
Question: {input}
""")

output_parser = StrOutputParser()

In [40]:
llm = ChatOllama(model = os.getenv("OLLAMA_MODEL"))
llm

ChatOllama(model='llama3:latest')

In [41]:
from langchain_core.runnables import RunnablePassthrough

document_chain = prompt | llm | output_parser

In [42]:
document_chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the context below:\n\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
| ChatOllama(model='llama3:latest')
| StrOutputParser()

In [43]:
from langchain_core.documents import Document

In [44]:
document_chain.invoke({
    "input": query,
    "context": [Document(page_content="Having observability set up from the start lets you iterate faster. You can see exactly what is being sent to the model, what is coming back, and where time is being spent, without adding print statements or running a debugger.")]
})

'According to the context, having observability set up from the start lets you iterate faster by:\n\n* Seeing exactly what is being sent to the model\n* Seeing exactly what is coming back\n* Identifying where time is being spent\n\nIt can be set up without adding print statements or running a debugger.'

#### What’s Really Happening Under the Hood: What I want to implement

```python
query = "What is LangChain?"

# Step 1: Retrieve docs
docs = retriever.get_relevant_documents(query)

# Step 2: Format prompt
context = "\n\n".join([doc.page_content for doc in docs])

final_prompt = f"""
Answer using only this:

{context}

Question: {query}
"""

# Step 3: LLM generates answer
llm(final_prompt)
```

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever()

retriever.invoke(query)

[Document(id='a3af3819-d653-4e03-a00e-4c713702895e', metadata={'language': 'en', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain'}, page_content='\u200bA/B testing\nGroup-by functionality requires at least two different values for a given metadata key.\nBecause you have been logging the llm metadata attribute, you can group monitoring charts by that attribute to compare model performance over time. From Monitoring in the UI sidebar, click Group by in the top left corner, select Metadata from the dropdown, then select llm. The charts update to show results grouped by that attribute. For more on grouping and custom charts, refer to Dashboards.\n\u200bDrilldown\nWhen a monitoring chart shows something unexpected, click a data point to freeze the tooltip, then click the metric name 

In [ ]:
from langchain_core.runnables import RunnableLambda
import re

def format_doc(doc):
    return ("\n\n".join(doc.page_content))

In [49]:
from langchain_core.runnables import RunnableLambda

retriever_rag = retriever | RunnableLambda(format_docs)
print(retriever_rag.invoke(query))

​A/B testing
Group-by functionality requires at least two different values for a given metadata key.
Because you have been logging the llm metadata attribute, you can group monitoring charts by that attribute to compare model performance over time. From Monitoring in the UI sidebar, click Group by in the top left corner, select Metadata from the dropdown, then select llm. The charts update to show results grouped by that attribute. For more on grouping and custom charts, refer to Dashboards.
​Drilldown
When a monitoring chart shows something unexpected, click a data point to freeze the tooltip, then click the metric name (for example, Input) to jump to the filtered runs table for that time window. For more on searching and filtering runs, refer to Filter traces.

if __name__ == "__main__":
    support_bot("How many users can I have on the Starter plan?")

Both metadata values appear on the trace. You can filter runs by metadata using the filtering controls in the Runs table.
​Productio

In [50]:
retriever_rag

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000171187956D0>, search_kwargs={})
| RunnableLambda(format_docs)

In [51]:
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "input": RunnablePassthrough()
    }
    | document_chain
)

rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000171187956D0>, search_kwargs={})
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the context below:\n\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
| ChatOllama(model='llama3:latest')
| StrOutputParser()

In [52]:
rag_chain.invoke(query)

'According to the provided context, having strong observability in place provides a benefit of being able to confidently ship to production. This is because you have significantly more traffic in production and can\'t examine every trace individually. With monitoring tools, you can understand aggregate behavior and drill down when something looks wrong.\n\nTo set up observability, it is not explicitly stated how to do so from the context provided. However, it appears that you would need to log metadata attributes (such as the "llm" attribute) in order to group monitoring charts by that attribute and compare model performance over time.'

In [53]:
print(rag_chain.invoke(query))

Based on the context, having observability set up from the start provides the ability to "confidently ship to production". This means that with strong observability in place, you can deploy your application to a live environment without worrying about being able to debug issues that may arise.

According to the context, observability can be set up by logging the `llm` metadata attribute and then grouping monitoring charts by this attribute.


In [54]:
response = rag_chain.invoke(query)
print(response)

Based on the provided context, having observability set up from the start provides strong confidence in shipping to production. This is because observability tools help understand aggregate behavior and allow for drilling down when something looks wrong.

As for setting up observability, there is no specific information provided on how to do it. However, it can be inferred that it involves logging metadata attributes, such as llm, which allows for grouping monitoring charts by that attribute to compare model performance over time.


In [61]:
test = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "input": RunnablePassthrough()
    }
)
test

{'context': VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000171187956D0>, search_kwargs={})
 | RunnableLambda(format_docs),
 'input': RunnablePassthrough()}

In [67]:
my_dict = {
    "context": "test",
    "input": "input1"
}

s = lambda x: x["context"] + " " + x['input']
s(my_dict)

'test input1'

In [70]:
from langchain_core.runnables import RunnableLambda

chain = RunnableLambda(lambda x: x.upper())

chain.invoke("hello")

'HELLO'

In [57]:
final_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "input": RunnablePassthrough()
    }
    | RunnableLambda(lambda x: {
        "input": x["input"],
        "context": x["context"],
        "answer": document_chain.invoke(x)
    })
)

response = final_chain.invoke(query)
print(response)

{'input': 'What benefit does having observability set up from the start has and how can it be set up', 'context': '\u200bA/B testing\nGroup-by functionality requires at least two different values for a given metadata key.\nBecause you have been logging the llm metadata attribute, you can group monitoring charts by that attribute to compare model performance over time. From Monitoring in the UI sidebar, click Group by in the top left corner, select Metadata from the dropdown, then select llm. The charts update to show results grouped by that attribute. For more on grouping and custom charts, refer to Dashboards.\n\u200bDrilldown\nWhen a monitoring chart shows something unexpected, click a data point to freeze the tooltip, then click the metric name (for example, Input) to jump to the filtered runs table for that time window. For more on searching and filtering runs, refer to Filter traces.\n\nif __name__ == "__main__":\n    support_bot("How many users can I have on the Starter plan?")\n

In [58]:
response

{'input': 'What benefit does having observability set up from the start has and how can it be set up',
 'context': '\u200bA/B testing\nGroup-by functionality requires at least two different values for a given metadata key.\nBecause you have been logging the llm metadata attribute, you can group monitoring charts by that attribute to compare model performance over time. From Monitoring in the UI sidebar, click Group by in the top left corner, select Metadata from the dropdown, then select llm. The charts update to show results grouped by that attribute. For more on grouping and custom charts, refer to Dashboards.\n\u200bDrilldown\nWhen a monitoring chart shows something unexpected, click a data point to freeze the tooltip, then click the metric name (for example, Input) to jump to the filtered runs table for that time window. For more on searching and filtering runs, refer to Filter traces.\n\nif __name__ == "__main__":\n    support_bot("How many users can I have on the Starter plan?")\

In [69]:
print(response["answer"])

According to the context, having observability set up from the start provides strong observability that allows you to:

* Understand aggregate behavior in production (where there is a large volume of traffic)
* Drill down when something looks wrong

To set up observability, you would need to follow the steps mentioned earlier, which involve:

1. Logging metadata attributes (in this case, the `llm` attribute was logged)
2. Using the Group-by functionality in the UI to group monitoring charts by the `llm` attribute
3. Using Drilldown to examine individual traces and filter runs by metadata

By setting up observability from the start, you can gain visibility into your system's behavior and make data-driven decisions about how to optimize it.


In [71]:
from langchain_core.runnables import RunnableParallel

In [72]:
final_chain = (
    RunnableParallel(
        input=RunnablePassthrough(),
        docs=retriever
    )
    | RunnableLambda(lambda x: {
        "input": x["input"],
        "docs": x["docs"],  # RAW Document objects
        "context": format_docs(x["docs"]),  # string version
        "answer": document_chain.invoke({
            "input": x["input"],
            "context": format_docs(x["docs"])
        })
    })
)

In [73]:
response = final_chain.invoke(query)
response

{'input': 'What benefit does having observability set up from the start has and how can it be set up',
 'docs': [Document(id='a3af3819-d653-4e03-a00e-4c713702895e', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'language': 'en', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'title': 'Trace an LLM application tutorial - Docs by LangChain'}, page_content='\u200bA/B testing\nGroup-by functionality requires at least two different values for a given metadata key.\nBecause you have been logging the llm metadata attribute, you can group monitoring charts by that attribute to compare model performance over time. From Monitoring in the UI sidebar, click Group by in the top left corner, select Metadata from the dropdown, then select llm. The charts update to show results grouped by that attribute. For more on grouping and custom charts, refer to Dashboards.\n\u200bDrilldown\nWhen a mo

In [74]:
response['docs']

[Document(id='a3af3819-d653-4e03-a00e-4c713702895e', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'language': 'en', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'title': 'Trace an LLM application tutorial - Docs by LangChain'}, page_content='\u200bA/B testing\nGroup-by functionality requires at least two different values for a given metadata key.\nBecause you have been logging the llm metadata attribute, you can group monitoring charts by that attribute to compare model performance over time. From Monitoring in the UI sidebar, click Group by in the top left corner, select Metadata from the dropdown, then select llm. The charts update to show results grouped by that attribute. For more on grouping and custom charts, refer to Dashboards.\n\u200bDrilldown\nWhen a monitoring chart shows something unexpected, click a data point to freeze the tooltip, then click the metric name 

In [ ]:
def build_output(x):
    docs = x["docs"]
    return {
        "input": x["input"],
        "docs": docs,
        "context": format_docs(docs),
        "answer": document_chain.invoke({
            "input": x["input"],
            "context": format_docs(docs)
        })
    }

final_chain = (
    RunnableParallel(
        input=RunnablePassthrough(),
        docs=retriever
    )
    | RunnableLambda(build_output)
)